# 03. Контактная политика и ROI

**Цель**  
Построить правила коммуникации на основе **uplift-score** (а не просто propensity) и оценить экономику таргетинга.

**Логика политики**
- **Активно** — высокий положительный uplift (persuadables)
- **Стандарт** — около нуля
- **Пауза** — отрицательный uplift (sleeping dogs / sure things, которых лучше не трогать)

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

PROJECT_ROOT = Path().resolve().parents[0]
sys.path.insert(0, str(PROJECT_ROOT))

from src.policy import (
    assign_contact_policy,
    evaluate_segments_by_top_fraction,
    policy_summary,
)
from src.utils import calculate_segment_economics

plt.rcParams["figure.figsize"] = (10, 5)

## 1. Загрузка предсказаний uplift

In [ ]:
data = joblib.load(PROJECT_ROOT / "outputs" / "prepared_data.joblib")
uplift_test = np.load(PROJECT_ROOT / "outputs" / "uplift_test.npy")

y_test = data["y_test"]
t_test = data["t_test"]

print(f"Test size: {len(uplift_test)}")
print(f"Mean uplift: {uplift_test.mean():.4f}")

## 2. Назначение контактной политики

In [ ]:
# Пороги можно калибровать под бизнес-ограничения
HIGH_THRESHOLD = 0.02   # выше → Активно
LOW_THRESHOLD = -0.01   # ниже → Пауза

policy = assign_contact_policy(
    uplift_test,
    high_threshold=HIGH_THRESHOLD,
    low_threshold=LOW_THRESHOLD,
)

summary = policy_summary(policy)
print(summary.round(3))

summary["count"].plot(kind="bar", color=["#2ecc71", "#3498db", "#e74c3c"])
plt.title("Распределение контактной политики")
plt.ylabel("Количество клиентов")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "03_policy_distribution.png", dpi=120, bbox_inches="tight")
plt.show()

## 3. Оценка экономики по top-fraction (ключевой блок)

In [1]:
# (можно менять)
MARGIN_PER_INCREMENTAL_CONVERSION = 25.0   # маржа с одной доп. конверсии
COST_PER_CONTACT = 2.0                    # стоимость одного контакта

segment_results = evaluate_segments_by_top_fraction(
    y_true=y_test,
    treatment=t_test,
    uplift_scores=uplift_test,
    fractions=[0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.70, 1.00],
    margin_per_conversion=MARGIN_PER_INCREMENTAL_CONVERSION,
    cost_per_contact=COST_PER_CONTACT,
)

segment_results["top_fraction_pct"] = (segment_results["top_fraction"] * 100).astype(int)
print(segment_results.round(4).to_string(index=False))

NameError: name 'evaluate_segments_by_top_fraction' is not defined

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(segment_results["top_fraction_pct"], segment_results["observed_uplift"], marker="o")
axes[0].axhline(0, color="gray", linestyle="--")
axes[0].set_xlabel("Top % of population")
axes[0].set_ylabel("Observed uplift")
axes[0].set_title("Observed uplift by targeting depth")
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    segment_results["top_fraction_pct"],
    segment_results["estimated_incremental_profit"],
    marker="o",
    color="green",
)
axes[1].axhline(0, color="gray", linestyle="--")
axes[1].set_xlabel("Top % of population")
axes[1].set_ylabel("Estimated incremental profit")
axes[1].set_title("Profit vs targeting depth")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "figures" / "03_economics_by_depth.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Детальный расчёт для выбранного сегмента

In [ ]:
TOP_FRACTION = 0.20   # например, топ-20%

# Надёжный поиск ближайшего значения 
diffs = (segment_results["top_fraction"] - TOP_FRACTION).abs()
closest_idx = diffs.idxmin()

if diffs.loc[closest_idx] > 1e-6:
    print(
        f"Предупреждение: ближайший top_fraction = "
        f"{segment_results.loc[closest_idx, 'top_fraction']:.4f} "
        f"(искали {TOP_FRACTION})"
    )

row = segment_results.loc[closest_idx]

if pd.isna(row["observed_uplift"]):
    print("Нельзя оценить экономику: в выбранном сегменте нет обеих групп.")
else:
    n_selected = int(row["n_selected"])
    incremental_conversions = n_selected * row["observed_uplift"]
    estimated_incremental_profit = (
        incremental_conversions * MARGIN_PER_INCREMENTAL_CONVERSION
        - n_selected * COST_PER_CONTACT
    )

    print(f"Сегмент: top {TOP_FRACTION:.0%}")
    print(f"Отобрано клиентов: {n_selected:,}")
    print(f"Observed uplift: {row['observed_uplift']:.4f}")
    print(f"Оценка дополнительных конверсий: {incremental_conversions:.2f}")
    print(f"Оценка дополнительной прибыли: {estimated_incremental_profit:,.2f}")

## 5. Сравнение политик (ROI simulation)

In [ ]:
def simulate_profit(selected_mask, y, treatment, margin=25.0, cost=2.0):
    """Приближённая оценка: считаем observed uplift только на выбранных."""
    if selected_mask.sum() == 0:
        return 0.0, 0.0
    y_s = y[selected_mask]
    t_s = treatment[selected_mask]
    if t_s.sum() == 0 or (1 - t_s).sum() == 0:
        return 0.0, 0.0
    uplift = y_s[t_s == 1].mean() - y_s[t_s == 0].mean()
    n = selected_mask.sum()
    inc_conv = n * uplift
    profit = inc_conv * margin - n * cost
    return inc_conv, profit


# Политика «Активно»
mask_active = policy == "Активно"
inc_a, profit_a = simulate_profit(mask_active, y_test, t_test,
                                  MARGIN_PER_INCREMENTAL_CONVERSION, COST_PER_CONTACT)

# Топ-20% по uplift
top20_n = int(len(uplift_test) * 0.20)
mask_top20 = np.zeros(len(uplift_test), dtype=bool)
mask_top20[np.argsort(-uplift_test)[:top20_n]] = True
inc_t, profit_t = simulate_profit(mask_top20, y_test, t_test,
                                  MARGIN_PER_INCREMENTAL_CONVERSION, COST_PER_CONTACT)

# Вся популяция (random / everyone)
inc_all, profit_all = simulate_profit(
    np.ones(len(uplift_test), dtype=bool), y_test, t_test,
    MARGIN_PER_INCREMENTAL_CONVERSION, COST_PER_CONTACT
)

comparison = pd.DataFrame([
    {"strategy": "Активно (policy)", "n": mask_active.sum(), "inc_conv": inc_a, "profit": profit_a},
    {"strategy": "Top-20% uplift", "n": mask_top20.sum(), "inc_conv": inc_t, "profit": profit_t},
    {"strategy": "Everyone", "n": len(uplift_test), "inc_conv": inc_all, "profit": profit_all},
])
print(comparison.round(2).to_string(index=False))

In [ ]:
# Сохраняем результаты
segment_results.to_csv(PROJECT_ROOT / "outputs" / "segment_economics.csv", index=False)

policy_df = pd.DataFrame({
    "idx": data["idx_test"],
    "uplift_score": uplift_test,
    "policy": policy,
    "y_true": y_test,
    "treatment": t_test,
})
policy_df.to_csv(PROJECT_ROOT / "outputs" / "contact_policy_test.csv", index=False)

print("Saved: outputs/segment_economics.csv, outputs/contact_policy_test.csv")

## Выводы

- Контактная политика теперь опирается на **uplift**, а не на сырую вероятность покупки.
- Таблица `segment_results` показывает, на какой глубине таргетинга достигается максимальная прибыль.
- Рекомендация: выбирать `top_fraction`, где `estimated_incremental_profit` максимален при приемлемом объёме контактов.
- Константы `MARGIN` и `COST` нужно калибровать под реальный бизнес.